# 05. Pipeline y Torneo de Algoritmos (Dataset en Español)

En este cuaderno vamos a replicar la metodología del modelo inglés, pero aplicando las técnicas de NLP sobre el texto traducido. El objetivo final es generar el histórico de métricas (`tracker_es.csv`) para ejecutar el Test A/B comparativo de la tesis.

### Paso 1: Carga de datos y Control de Caché

El proceso de traducción del dataset (23.117 tickets) mediante el modelo `Helsinki-NLP/opus-mt-en-es` de Hugging Face consume alrededor de 12 horas por CPU. 
Para optimizar el código y hacerlo reproducible, hemos implementado una comprobación previa. Si el archivo procesado ya existe en disco, se cargará directamente. Si no existe, el código aislará el entorno de Hugging Face y ejecutará la traducción por lotes (*batching*) apagando los gradientes de la red para ahorrar memoria RAM.

In [1]:
import os
import pandas as pd

# Definimos las rutas relativas
ruta_silver = "../data/processed/df_final_silver.parquet"
ruta_traduccion = "../data/processed/df_final_silver_es.parquet"

# CONTROL DE CACHÉ
if os.path.exists(ruta_traduccion):
    print("✅ Caché detectado: Archivo parquet localizado en disco.")
    print("⏭️ Omitiendo fase de inferencia neuronal. Cargando datos en memoria...")
    
    # Cargamos directamente el resultado final
    df_espanol = pd.read_parquet(ruta_traduccion)
    print(f"Dataset cargado correctamente: {df_espanol.shape[0]} tickets listos para procesar.")

else:
    print("⚠️ Caché no localizado. Iniciando instancia de MarianMT (Hugging Face)...")
    print("⏳ Este proceso computacional depende del volumen de filas del dataset introducido.")
    
    # Importaciones condicionales (solo se cargan si hace falta traducir)
    import torch
    from transformers import MarianMTModel, MarianTokenizer
    from tqdm.auto import tqdm
    
    # 1. Configurar caché de modelos en el entorno virtual (.venv)
    current_dir = os.getcwd()
    cache_path = os.path.abspath(os.path.join(current_dir, "..", ".venv", "huggingface_cache"))
    os.makedirs(cache_path, exist_ok=True)
    os.environ['HF_HOME'] = cache_path
    
    # 2. Cargar el dataset original y preparar textos
    df_silver = pd.read_parquet(ruta_silver)
    df_silver['ticket_id'] = ['TKT-' + str(i).zfill(5) for i in range(1, len(df_silver) + 1)]
    # Mantenemos mayúsculas porque el modelo de traducción las necesita para el contexto
    df_silver['full_text'] = (df_silver['subject'] + " " + df_silver['body']).str.strip()
    
    # Separamos el idioma
    df_nativo_es = df_silver[df_silver['language'] == 'es'].copy()
    df_to_translate = df_silver[df_silver['language'] == 'en'][['ticket_id', 'full_text']].copy()
    
    # 3. Preparar modelo Hugging Face
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model_name = "Helsinki-NLP/opus-mt-en-es"
    tokenizer = MarianTokenizer.from_pretrained(model_name)
    model = MarianMTModel.from_pretrained(model_name).to(device)
    
    # 4. Bucle de traducción por lotes (Batching)
    BATCH_SIZE = 32
    textos_en_ingles = df_to_translate['full_text'].tolist()
    ids_en_ingles = df_to_translate['ticket_id'].tolist()
    translated_data = []
    
    for i in tqdm(range(0, len(textos_en_ingles), BATCH_SIZE), desc="Traduciendo Tickets"):
        batch_texts = textos_en_ingles[i : i + BATCH_SIZE]
        batch_ids = ids_en_ingles[i : i + BATCH_SIZE]
        
        # Tokenización con recorte de seguridad (max 512) para no saturar memoria
        inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
        
        # Apagamos los gradientes matemáticos porque no estamos entrenando
        with torch.no_grad():
            translated_tokens = model.generate(**inputs)
            
        translated_batch = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)
        
        # Guardamos el texto emparejado con su ID original
        for t_id, t_text in zip(batch_ids, translated_batch):
            translated_data.append({"ticket_id": t_id, "full_text_es": t_text})
            
    # 5. Ensamblaje y guardado final
    df_traducido = pd.DataFrame(translated_data)
    df_es_fusion = pd.merge(df_silver[df_silver['language'] == 'en'], df_traducido, on='ticket_id', how='inner')
    df_es_fusion['full_text'] = df_es_fusion['full_text_es']
    df_es_fusion = df_es_fusion.drop(columns=['full_text_es'])
    
    df_espanol = pd.concat([df_es_fusion, df_nativo_es], ignore_index=True)
    df_espanol = df_espanol.drop(columns=['subject', 'body'])
    
    df_espanol.to_parquet(ruta_traduccion, index=False)
    print(f"\n✅ Traducción completada y guardada en: {ruta_traduccion}")

✅ Caché detectado: Archivo parquet localizado en disco.
⏭️ Omitiendo fase de inferencia neuronal. Cargando datos en memoria...
Dataset cargado correctamente: 23867 tickets listos para procesar.


### Paso 2: Creación de la Tripleta y NLP (Vectorización TF-IDF)

En esta fase preparamos los datos para que los algoritmos puedan procesarlos. Primero unimos las variables de negocio (`queue`, `type`, `priority`) para recrear la variable objetivo de 84 clases (La Tripleta) y aseguramos la estratificación en el corte de Train (80%) y Test (20%).

A continuación, cargamos el modelo estadístico de spaCy específico para el español (`es_core_news_sm`). Este modelo se encargará de lematizar los textos (llevar los verbos a su infinitivo) y eliminar las palabras vacías (stop words). Finalmente, vectorizamos los textos limpios usando TF-IDF, limitando la matriz a 10.000 palabras clave.

In [2]:
import spacy
import subprocess
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

print("--- FASE 2: INGENIERÍA Y VECTORIZACIÓN (ESPAÑOL) ---\n")

# 1. Limpieza básica de seguridad
df_espanol.dropna(subset=['queue', 'type', 'priority', 'full_text'], inplace=True)

# 2. Creación de la variable objetivo
df_espanol['target_tripleta'] = df_espanol['queue'] + " - " + df_espanol['type'] + " - " + df_espanol['priority']

# Filtramos clases que tengan menos de 2 tickets (requisito matemático para poder separar Train/Test)
conteo = df_espanol['target_tripleta'].value_counts()
clases_validas = conteo[conteo > 1].index
df_espanol = df_espanol[df_espanol['target_tripleta'].isin(clases_validas)]

# 3. Separación de datos
X = df_espanol['full_text']
y = df_espanol['target_tripleta']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
print(f"✅ Split realizado. Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

# 4. Carga del cerebro NLP en Español
# Usamos try/except por si es la primera vez que lo corres y no está instalado en el .venv
try:
    nlp_es = spacy.load('es_core_news_sm')
except OSError:
    print("⏳ Descargando diccionario spaCy en español (es_core_news_sm)...")
    subprocess.run(["python", "-m", "spacy", "download", "es_core_news_sm"])
    nlp_es = spacy.load('es_core_news_sm')

# 5. Función de Lematización
def limpiar_texto_es(texto):
    # Procesamos el texto en minúsculas
    doc = nlp_es(texto.lower())
    # Guardamos el lema (raíz) si no es stop word, ni puntuación, ni un número aislado
    tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct and not token.like_num]
    return " ".join(tokens)

print("⏳ Iniciando limpieza NLP con spaCy (esto puede tardar unos 2 minutos)...")
X_train_limpio = X_train.apply(limpiar_texto_es)
X_test_limpio = X_test.apply(limpiar_texto_es)
print("✅ Limpieza NLP terminada.")

# 6. Transformación Matemática (TF-IDF)
print("⏳ Vectorizando con TF-IDF (Límite: 10.000 palabras)...")
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2))

# Ajustamos y transformamos Train
X_train_tfidf = tfidf.fit_transform(X_train_limpio)
# Solo transformamos Test (sin ajustar, para evitar data leakage)
X_test_tfidf = tfidf.transform(X_test_limpio)

print(f"✅ Matrices generadas:")
print(f"   -> Dimensiones Train: {X_train_tfidf.shape}")
print(f"   -> Dimensiones Test:  {X_test_tfidf.shape}")

--- FASE 2: INGENIERÍA Y VECTORIZACIÓN (ESPAÑOL) ---

✅ Split realizado. Train: 19093 | Test: 4774
⏳ Iniciando limpieza NLP con spaCy (esto puede tardar unos 2 minutos)...
✅ Limpieza NLP terminada.
⏳ Vectorizando con TF-IDF (Límite: 10.000 palabras)...
✅ Matrices generadas:
   -> Dimensiones Train: (19093, 10000)
   -> Dimensiones Test:  (4774, 10000)


### Paso 3: Torneo de Algoritmos (Baseline sin balanceo)

Antes de alterar la distribución de los datos, establecemos un *baseline* (punto de referencia). Primero, transformamos las 84 etiquetas de texto de la Tripleta a valores numéricos mediante `LabelEncoder` (requisito técnico para los algoritmos de Boosting). 

A continuación, inicializamos el panel con 6 familias algorítmicas diferentes y las entrenamos sobre la matriz original en español, que está altamente desbalanceada. Esto nos demostrará qué algoritmos colapsan ante las clases raras y cuáles son capaces de generalizar por sí mismos.

In [3]:
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, precision_score, recall_score
import time
import pandas as pd

print("--- FASE 3: ENTRENAMIENTO SIN BALANCEAR (ESPAÑOL) ---\n")

# 1. Transformación de Etiquetas (Texto a Números)
encoder = LabelEncoder()
y_train_encoded = encoder.fit_transform(y_train)
y_test_encoded = encoder.transform(y_test)

# 2. Definición del panel de modelos (silenciando advertencias de convergencia)
modelos = {
    "Naive Bayes": MultinomialNB(),
    "Linear SVC": LinearSVC(random_state=42, max_iter=2000),
    "Regresion Logistica": LogisticRegression(random_state=42, max_iter=2000),
    "Random Forest": RandomForestClassifier(random_state=42, n_jobs=-1),
    "LightGBM": LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1),
    "XGBoost": XGBClassifier(random_state=42, n_jobs=-1, eval_metric='mlogloss', verbosity=0)
}

# 3. Bucle de Entrenamiento
resultados_sin_smote = []

for nombre, modelo in modelos.items():
    print(f"⏳ Entrenando {nombre} (Sin SMOTE)...")
    start_time = time.time()
    
    # Entrenamos sobre los datos puros
    modelo.fit(X_train_tfidf, y_train_encoded)
    tiempo_entrenamiento = round(time.time() - start_time, 2)
    
    # Evaluamos en el bloque de examen
    y_pred = modelo.predict(X_test_tfidf)
    
    f1_macro = round(f1_score(y_test_encoded, y_pred, average='macro', zero_division=0), 4)
    f1_micro = round(f1_score(y_test_encoded, y_pred, average='micro', zero_division=0), 4)
    precision = round(precision_score(y_test_encoded, y_pred, average='macro', zero_division=0), 4)
    recall = round(recall_score(y_test_encoded, y_pred, average='macro', zero_division=0), 4)
    
    resultados_sin_smote.append({
        'Modelo': nombre, 
        'Condicion': 'Sin SMOTE', 
        'F1-Macro': f1_macro,
        'F1-Micro': f1_micro, 
        'Precision (Macro)': precision, 
        'Recall (Macro)': recall,
        'Tiempo (seg)': tiempo_entrenamiento
    })
    
    print(f"✅ Completado -> F1-Macro: {f1_macro} | Tiempo: {tiempo_entrenamiento}s")

# Guardamos el bloque en un DataFrame temporal
df_resultados_sin_smote = pd.DataFrame(resultados_sin_smote)

--- FASE 3: ENTRENAMIENTO SIN BALANCEAR (ESPAÑOL) ---

⏳ Entrenando Naive Bayes (Sin SMOTE)...
✅ Completado -> F1-Macro: 0.0308 | Tiempo: 0.11s
⏳ Entrenando Linear SVC (Sin SMOTE)...
✅ Completado -> F1-Macro: 0.5738 | Tiempo: 17.59s
⏳ Entrenando Regresion Logistica (Sin SMOTE)...
✅ Completado -> F1-Macro: 0.1599 | Tiempo: 22.52s
⏳ Entrenando Random Forest (Sin SMOTE)...
✅ Completado -> F1-Macro: 0.5982 | Tiempo: 8.41s
⏳ Entrenando LightGBM (Sin SMOTE)...
✅ Completado -> F1-Macro: 0.0072 | Tiempo: 219.38s
⏳ Entrenando XGBoost (Sin SMOTE)...
✅ Completado -> F1-Macro: 0.4115 | Tiempo: 1078.55s


### Paso 4: Balanceo (SMOTE) y Exportación del Tracker

Tras evaluar el *baseline*, aplicamos SMOTE (*Synthetic Minority Over-sampling Technique*). Este método analiza geométricamente los clústeres de las clases minoritarias e inyecta tickets sintéticos hasta que las 84 clases tienen el mismo volumen de datos.

Volvemos a entrenar a los 6 contendientes sobre esta nueva matriz gigante, lo que teóricamente debería forzar a los algoritmos a prestar atención a las clases marginales, elevando su métrica de *Recall*. Finalmente, unimos ambas tablas y generamos el `tracker_es.csv` definitivo para nuestro Test A/B.

In [4]:
from imblearn.over_sampling import SMOTE

print("--- FASE 4: BALANCEO (SMOTE) Y ENTRENAMIENTO (ESPAÑOL) ---\n")

# 1. Aplicamos SMOTE a la matriz TF-IDF
print("⏳ Aplicando SMOTE a la matriz de 10.000 dimensiones...")
start_smote = time.time()

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote_encoded = smote.fit_resample(X_train_tfidf, y_train_encoded)

print(f"✅ SMOTE completado en {round(time.time() - start_smote, 2)}s.")
print(f"Nuevas dimensiones de Train: {X_train_smote.shape[0]} tickets (Clases igualadas)\n")

# 2. Bucle de Entrenamiento (Con datos inyectados)
resultados_con_smote = []

for nombre, modelo in modelos.items():
    print(f"⏳ Entrenando {nombre} (Con SMOTE)...")
    start_time = time.time()
    
    # Entrenamos sobre los datos matemáticamente aplanados
    modelo.fit(X_train_smote, y_train_smote_encoded)
    tiempo_entrenamiento = round(time.time() - start_time, 2)
    
    # Evaluamos en el bloque de examen original (nunca se le hace SMOTE al Test)
    y_pred = modelo.predict(X_test_tfidf)
    
    f1_macro = round(f1_score(y_test_encoded, y_pred, average='macro', zero_division=0), 4)
    f1_micro = round(f1_score(y_test_encoded, y_pred, average='micro', zero_division=0), 4)
    precision = round(precision_score(y_test_encoded, y_pred, average='macro', zero_division=0), 4)
    recall = round(recall_score(y_test_encoded, y_pred, average='macro', zero_division=0), 4)
    
    resultados_con_smote.append({
        'Modelo': nombre, 
        'Condicion': 'Con SMOTE', 
        'F1-Macro': f1_macro,
        'F1-Micro': f1_micro, 
        'Precision (Macro)': precision, 
        'Recall (Macro)': recall,
        'Tiempo (seg)': tiempo_entrenamiento
    })
    
    print(f"✅ Completado -> F1-Macro: {f1_macro} | Tiempo: {tiempo_entrenamiento}s")

# 3. Consolidación y Exportación del Tracker Español
df_resultados_con_smote = pd.DataFrame(resultados_con_smote)
df_tracker_final = pd.concat([df_resultados_sin_smote, df_resultados_con_smote], ignore_index=True)

df_tracker_final.to_csv("../data/processed/tracker_es.csv", index=False)
print("\n--- TRACKER ESPAÑOL EXPORTADO A DISCO ---")

# Mostramos la tabla final ordenada para poder compararla visualmente
display(df_tracker_final.sort_values(by=['Modelo', 'Condicion']))

--- FASE 4: BALANCEO (SMOTE) Y ENTRENAMIENTO (ESPAÑOL) ---

⏳ Aplicando SMOTE a la matriz de 10.000 dimensiones...
✅ SMOTE completado en 0.76s.
Nuevas dimensiones de Train: 159348 tickets (Clases igualadas)

⏳ Entrenando Naive Bayes (Con SMOTE)...
✅ Completado -> F1-Macro: 0.3758 | Tiempo: 0.66s
⏳ Entrenando Linear SVC (Con SMOTE)...
✅ Completado -> F1-Macro: 0.5884 | Tiempo: 123.41s
⏳ Entrenando Regresion Logistica (Con SMOTE)...
✅ Completado -> F1-Macro: 0.521 | Tiempo: 140.02s
⏳ Entrenando Random Forest (Con SMOTE)...
✅ Completado -> F1-Macro: 0.6658 | Tiempo: 185.5s
⏳ Entrenando LightGBM (Con SMOTE)...
✅ Completado -> F1-Macro: 0.0002 | Tiempo: 1965.13s
⏳ Entrenando XGBoost (Con SMOTE)...
✅ Completado -> F1-Macro: 0.3855 | Tiempo: 5178.65s

--- TRACKER ESPAÑOL EXPORTADO A DISCO ---


,Modelo,Condicion,F1-Macro,F1-Micro,Precision (Macro),Recall (Macro),Tiempo (seg)
10,LightGBM,Con SMOTE,0.0002,0.0054,0.0001,0.0115,1965.13
4,LightGBM,Sin SMOTE,0.0072,0.0522,0.0156,0.0159,219.38
7,Linear SVC,Con SMOTE,0.5884,0.5253,0.6202,0.5807,123.41
1,Linear SVC,Sin SMOTE,0.5738,0.5186,0.6632,0.5271,17.59
6,Naive Bayes,Con SMOTE,0.3758,0.3312,0.3528,0.4731,0.66
0,Naive Bayes,Sin SMOTE,0.0308,0.1783,0.0826,0.0450,0.11
9,Random Forest,Con SMOTE,0.6658,0.6280,0.7957,0.6002,185.50
3,Random Forest,Sin SMOTE,0.5982,0.5649,0.8171,0.5063,8.41
8,Regresion Logistica,Con SMOTE,0.5210,0.4346,0.5432,0.5243,140.02
2,Regresion Logistica,Sin SMOTE,0.1599,0.2989,0.3518,0.1477,22.52
